In [32]:
import os
from dotenv import load_dotenv
import requests
from agents import Agent, Runner, trace, function_tool, SQLiteSession
from openai.types.responses import ResponseTextDeltaEvent

load_dotenv(override=True)


True

### Create an instance of agent

In [14]:
agent = Agent(name="Jockester", instructions="You are a joke teller", model="gpt-5.4-nano")

In [15]:
result = await Runner.run(agent, "Tell me a good joke")

In [16]:
print(result.final_output)

Why don’t scientists trust atoms?

Because they make up everything.


In [17]:
result.to_input_list()

[{'content': 'Tell me a good joke', 'role': 'user'},
 {'id': 'msg_07f4dbdb8ed98c8f006ad50ded4e2887d29706b41d7fd63b48',
  'content': [{'annotations': [],
    'text': 'Why don’t scientists trust atoms?\n\nBecause they make up everything.',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'phase': 'final_answer'}]

### Observability `with trace():`

In [18]:
with trace("Telling Joke"):
    result = await Runner.run(agent, "Tell me a good joke")
print(result.final_output)

Why don’t skeletons fight each other?

They don’t have the guts.


### Streaming text

In [20]:
result = Runner.run_streamed(agent, "Tell me a brain tickling joke")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

I tried to tell a brain-tickling joke… but it kept going over my head.

In [26]:
result = Runner.run_streamed(agent, "Tell me the 5 best joke")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end = "", flush=True)

Sure—here are 5 best-ish jokes:

1) Why don’t scientists trust atoms?  
Because they make up everything!

2) Why did the scarecrow win an award?  
Because he was outstanding in his field!

3) What do you call fake spaghetti?  
An impasta!

4) Why can’t a bicycle stand up by itself?  
Because it’s two-tired!

5) Why did the math book look sad?  
Because it had too many problems!

### Adding tool

In [41]:
PUSHOVER_USER = os.getenv("PUSH_USER_KEY")
PUSHOVER_TOKEN = os.getenv("PUSH_API_TOKEN")
PUSHOVER_URL = "https://api.pushover.net/1/messages.json"
if PUSHOVER_TOKEN:
    if PUSHOVER_TOKEN.startswith('a'):
        print("Pushover Token Configured")
    else:
        print("Pushover token not configured")
else:
    print("Pushover user token not found")
if PUSHOVER_USER:
    if PUSHOVER_USER.startswith("u"):
        print("Pushover user configured")
    else:
        print("Pushover user not configured")
else:
    print("Pushover user not fount")

Pushover Token Configured
Pushover user configured


In [46]:
def push(message):
    print(f"Push: {message}")
    payload = {
        "user": PUSHOVER_USER,
        "token": PUSHOVER_TOKEN,
        "message": message 
    }
    requests.post(PUSHOVER_URL, data=payload)

In [47]:
push("Hey")

Push: Hey


In [54]:
@function_tool
def push_tool(message: str) -> str:
    """Send the message to the user as push notification"""
    print(f"Push: {message}")
    payload = {
        "user": PUSHOVER_USER,
        "token": PUSHOVER_TOKEN,
        "message": message
    }
    result = requests.post(url = PUSHOVER_URL, data=payload)
    return f"PUSH sent with api status code {result}"

In [56]:
push_tool.description

'Send the message to the user as push notification'

In [57]:
notifier = Agent(
        name="Notifier", 
        model='gpt-5.4-nano', 
        instructions="You notify the user when requested", 
        tools=[push_tool]
        )

In [64]:
with trace("Notifying the user"):
    result = await Runner.run(notifier, f"Notfiy the user with jokes")
print(result)
print(result.final_output)

Push: Why don’t scientists trust atoms? Because they make up everything. 😄
RunResult:
- Last agent: Agent(name="Notifier", ...)
- Final output (str):
    Why don’t scientists trust atoms?  
    Because they make up everything. 😄
- 3 new item(s)
- 2 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)
Why don’t scientists trust atoms?  
Because they make up everything. 😄
